# Day 37 — Practice Questions
### What Right Means — evaluation metrics for classification

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a report | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### Four questions to keep asking

- What is the **baseline**? What does doing nothing score?
- Of the four boxes, **which one is the expensive one**?
- Who chose the **cut-off**, and what does each mistake cost?
- Is this number a **rate out of the predictions**, or **out of the real cases**?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, fbeta_score,
                             balanced_accuracy_score, matthews_corrcoef,
                             roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             classification_report)

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("ready")

ready


---
# Part A — Work it out

Most of these need nothing but arithmetic. Work them out, then run the cell to check.

### A1 · Fill in the matrix

A spam filter was run on **1,000 emails**. 80 of them were really spam.
It flagged **95** emails, and **60** of those really were spam.

Draw the confusion matrix and work out all four boxes.

In [2]:
total, real_spam, flagged, tp = 1000, 80, 95, 60
fp = flagged - tp
fn = real_spam - tp
tn = total - tp - fp - fn
print(f"                 said legit   said spam")
print(f"actually legit   {tn:>10}   {fp:>9}")
print(f"actually spam    {fn:>10}   {tp:>9}")
print(f"\ncheck: {tn}+{fp}+{fn}+{tp} = {tn+fp+fn+tp}")

                 said legit   said spam
actually legit          885          35
actually spam            20          60

check: 885+35+20+60 = 1000


### A2 · The four metrics, by hand

From the matrix in A1, work out accuracy, precision, recall and specificity.
Then say what each sentence means in plain English.

In [3]:
tn, fp, fn, tp = 885, 35, 20, 60
print(f"accuracy    = ({tp} + {tn}) / 1000 = {(tp+tn)/1000:.4f}")
print(f"precision   = {tp} / ({tp} + {fp}) = {tp/(tp+fp):.4f}")
print(f"recall      = {tp} / ({tp} + {fn}) = {tp/(tp+fn):.4f}")
print(f"specificity = {tn} / ({tn} + {fp}) = {tn/(tn+fp):.4f}")
print(f"\nF1 = {2*(tp/(tp+fp))*(tp/(tp+fn))/((tp/(tp+fp))+(tp/(tp+fn))):.4f}")
print(f"'never flag anything' would score {920/1000:.4f} on accuracy")

accuracy    = (60 + 885) / 1000 = 0.9450
precision   = 60 / (60 + 35) = 0.6316
recall      = 60 / (60 + 20) = 0.7500
specificity = 885 / (885 + 35) = 0.9620

F1 = 0.6857
'never flag anything' would score 0.9200 on accuracy


### A3 · Precision or recall?

For each, say which of the two you would optimise, and in one line why.

1. A model that decides which luggage to open at airport security
2. A model that auto-deletes "obviously fake" job applications before any human sees them
3. A model that flags transactions for a human fraud analyst to review
4. A model that decides, with no human involved, whether to decline a card
5. A model that picks which 50 of 10,000 patients get a free preventive check-up

### A4 · Two models, one decision

| | precision | recall |
|---|---|---|
| **Model A** | 0.90 | 0.30 |
| **Model B** | 0.45 | 0.75 |

1. Work out F1 for each.
2. Which has the higher F1?
3. Now suppose this is a **cancer screen**. Which would you deploy?

In [4]:
for nm, p_, r_ in [("A", 0.90, 0.30), ("B", 0.45, 0.75)]:
    print(f"model {nm}:  F1 = {2*p_*r_/(p_+r_):.4f}   "
          f"F2 = {5*p_*r_/(4*p_+r_):.4f}   plain avg = {(p_+r_)/2:.3f}")

model A:  F1 = 0.4500   F2 = 0.3462   plain avg = 0.600
model B:  F1 = 0.5625   F2 = 0.6618   plain avg = 0.600


### A5 · What does AUC 0.80 mean?

Three statements. Which are true?

1. "The model is right 80% of the time."
2. "Pick a random positive and a random negative — there is an 80% chance the positive
   gets the higher score."
3. "At the best cut-off, accuracy is 80%."
4. "The model would still score 0.80 if the positive class were ten times rarer."

### A6 · Read a classification report

```
              precision    recall  f1-score   support

     routine     0.9400    0.9700    0.9548      3200
      urgent     0.4800    0.3100    0.3767       290
    critical     0.6700    0.2200    0.3312        50

    accuracy                         0.9140      3540
   macro avg     0.6967    0.5000    0.5542      3540
weighted avg     0.8886    0.9140    0.8972      3540
```

1. How many `critical` cases were there, and how many did the model find?
2. Why is `weighted avg` F1 so much higher than `macro avg`?
3. Which single row would you take to the meeting?

In [5]:
print("critical: support 50, recall 0.22  ->", round(50*0.22), "found,",
      50 - round(50*0.22), "missed")
print("urgent  : support 290, recall 0.31 ->", round(290*0.31), "found,",
      290 - round(290*0.31), "missed")
print()
print("routine is", f"{3200/3540:.1%}", "of all cases")

critical: support 50, recall 0.22  -> 11 found, 39 missed
urgent  : support 290, recall 0.31 -> 90 found, 200 missed

routine is 90.4% of all cases


### A7 · Cut-off arithmetic

A model scores 1,000 loan applications. 100 will default.

At a cut-off of **0.3** it flags 250 applications and catches 70 of the defaulters.
At a cut-off of **0.6** it flags 80 applications and catches 40 of the defaulters.

Work out precision and recall at each, and say which you would choose if a default costs
₹50,000 and wrongly refusing a good customer costs ₹3,000.

In [6]:
for t, flagged, caught in [(0.3, 250, 70), (0.6, 80, 40)]:
    fp_ = flagged - caught
    missed = 100 - caught
    cost = 50000*missed + 3000*fp_
    print(f"cut-off {t}:  precision {caught/flagged:.4f}   recall {caught/100:.4f}")
    print(f"             {missed} defaults missed = {50000*missed:>9,}")
    print(f"             {fp_} good customers refused = {3000*fp_:>9,}")
    print(f"             total = {cost:,}\n")
print(f"flag nobody: {50000*100:,}")

cut-off 0.3:  precision 0.2800   recall 0.7000
             30 defaults missed = 1,500,000
             180 good customers refused =   540,000
             total = 2,040,000

cut-off 0.6:  precision 0.5000   recall 0.4000
             60 defaults missed = 3,000,000
             40 good customers refused =   120,000
             total = 3,120,000

flag nobody: 5,000,000


### A8 · Which metric would you ask for?

Say what you would ask for, in one line each.

1. Your team has built 12 models and wants to shortlist 3 for proper testing.
2. A regulator asks whether your credit model treats two groups of applicants equally.
3. Your manager wants one number for a board slide.
4. You have to decide today whether to switch the fraud system on.

---
# Part B — Find the mistake

Four reports. Every number in them is **arithmetically correct**. Every one is
misleading.

In [7]:
# ---------------------------------------------------------------
# 3,000 hospital discharges. B and C both use `hosp`.
# Can we predict who will be readmitted within 30 days?
# ---------------------------------------------------------------
r = np.random.default_rng(3720)
N = 3000

age              = r.normal(63, 15, N).clip(20, 98).round(0)
days_in_hospital = r.gamma(2.2, 2.3, N).clip(1, 40).round(0)
prior_admissions = r.poisson(0.9, N).clip(0, 8).astype(float)
num_medications  = r.normal(13, 5.5, N).clip(1, 35).round(0)
discharge_to     = r.choice(["home", "home_care", "nursing"], N, p=[.58, .27, .15])

db = np.where(discharge_to == "nursing", 0.9,
     np.where(discharge_to == "home_care", 0.35, -0.25))
z = (-7.4 + 1.75*(0.022*age + 0.055*days_in_hospital + 0.52*prior_admissions
                  + 0.035*num_medications + db) + r.normal(0, 0.45, N))
readmitted = (r.uniform(0, 1, N) < sigmoid(z)).astype(int)

hosp = pd.DataFrame({"age": age, "days_in_hospital": days_in_hospital,
                     "prior_admissions": prior_admissions,
                     "num_medications": num_medications,
                     "discharge_to": discharge_to, "readmitted": readmitted})

NUM = ["age", "days_in_hospital", "prior_admissions", "num_medications"]
CAT = ["discharge_to"]

pipe = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), NUM),
                                ("cat", OneHotEncoder(drop="first"), CAT)])),
    ("model", LogisticRegression(max_iter=2000))])

X, y = hosp[NUM + CAT], hosp.readmitted
cv = StratifiedKFold(5, shuffle=True, random_state=0)
probs = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
pred  = (probs >= 0.5).astype(int)

print("rows:", len(hosp), "  readmitted:", int(y.sum()))
hosp.head()

rows: 3000   readmitted: 362


,age,days_in_hospital,prior_admissions,num_medications,discharge_to,readmitted
0,50.0,3.0,2.0,13.0,home,0
1,41.0,1.0,0.0,20.0,nursing,0
2,50.0,3.0,3.0,12.0,home,0
3,51.0,3.0,1.0,17.0,home,0
4,60.0,1.0,3.0,16.0,home,0


### B1 · "Nearly 90% accurate"

In [8]:
print(f"accuracy: {accuracy_score(y, pred):.4f}")
print()
print("REPORT: 'The readmission model is 89.3% accurate. We recommend")
print("         rolling it out across all wards.'")

accuracy: 0.8927

REPORT: 'The readmission model is 89.3% accurate. We recommend
         rolling it out across all wards.'


### B2 · "Precision of 0.70 — better than the published literature"

In [9]:
print(f"precision: {precision_score(y, pred):.4f}")
print()
print("REPORT: 'At 0.70, our precision exceeds the 0.55 reported in the")
print("         published literature for this task.'")

precision: 0.7000

REPORT: 'At 0.70, our precision exceeds the 0.55 reported in the
         published literature for this task.'


### B3 · "AUC 0.81 — strong discrimination"

In [10]:
print(f"ROC AUC: {roc_auc_score(y, probs):.4f}")
print()
print("REPORT: 'The model achieves an AUC of 0.81, indicating strong")
print("         discrimination. It is ready for clinical use.'")

ROC AUC: 0.8147

REPORT: 'The model achieves an AUC of 0.81, indicating strong
         discrimination. It is ready for clinical use.'


### B4 · "Weighted F1 of 0.86 across all discharge destinations"

In [11]:
print("REPORT: 'Performance is consistent across discharge destinations,")
print("         with a weighted F1 of 0.86.'")
print()
print(f"weighted F1: {f1_score(y, pred, average='weighted'):.4f}")

REPORT: 'Performance is consistent across discharge destinations,
         with a weighted F1 of 0.86.'



weighted F1: 0.8648


---
# Part C — Write the code

Five tasks on the same hospital data.

### C1 · Baseline and the four boxes

Print the readmission rate, the do-nothing baseline, the full confusion matrix at the
default 0.5, and all five of accuracy / precision / recall / F1 / balanced accuracy.
Also break the readmission rate down by `discharge_to`.

In [12]:
# your code here

### C2 · The threshold table

Build a table of flagged / caught / missed / precision / recall / F1 / accuracy for
cut-offs from 0.05 to 0.70, then plot precision and recall against the cut-off.
Say which cut-off maximises F1.

In [13]:
# your code here

### C3 · The two curves

Draw the ROC and precision–recall curves on the same figure, each labelled with its
summary number. Mark the no-skill line on both. Then say in two sentences what each
curve tells you that the other does not.

In [14]:
# your code here

### C4 · Choose the cut-off with money

The hospital tells you:

- a readmission costs them a penalty of **₹9,000**
- a follow-up care call costs **₹250**
- a call prevents about **40%** of the readmissions it reaches

Build the cost table, find the cheapest cut-off, and compare it with doing nothing.

In [15]:
# your code here

### C5 · Write the recommendation

Using everything above, write a short recommendation to the hospital's clinical director.
Not a methodology lecture — a decision, with the numbers that support it, and an honest
statement of what you are unsure about.

In [16]:
# your code here — produce whatever numbers you need to support it

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Compute AUC yourself, by counting pairs

The lecture claimed:

> **AUC is the probability that a randomly chosen positive scores higher than a randomly
> chosen negative.**

Prove it. Write **two** functions, using no `roc_auc_score`:

1. `auc_by_sampling(y, probs, n_pairs, seed)` — draw random (positive, negative) pairs and
   count how often the positive wins. Ties count as half.
2. `auc_exact(y, probs)` — compute it **exactly**, without sampling, by comparing every
   positive against every negative. Handle ties properly.

Then show that both land on `roc_auc_score`, and that the exact one is identical.

*Hint for the exact version: there is a way to do it in one pass using ranks, which is why
`roc_auc_score` is fast on a million rows. Try the straightforward way first.*

In [17]:
# your functions here

### D2 · Referee a disagreement

Two teams built models to flag suspicious insurance claims for investigation. Both used
the same 20,000 claims, the same split, and cross-validation. **2% of claims are
fraudulent.**

| | Team Alpha | Team Beta |
|---|---|---|
| accuracy | 0.9810 | 0.9240 |
| precision | 0.6400 | 0.1900 |
| recall | 0.3200 | 0.8100 |
| F1 | 0.4267 | 0.3078 |
| ROC AUC | 0.8900 | 0.9100 |

**Alpha says:** *"We win on accuracy, precision and F1. Three metrics out of five."*

**Beta says:** *"We win on AUC, and we catch 81% of the fraud instead of 32%."*

**Your tasks:**

1. Reconstruct both confusion matrices from the numbers given.
2. Work out who is right, and under what circumstances each is right.
3. The company says an investigation costs **₹4,000** and the average fraudulent claim is
   **₹85,000**. Settle it.
4. Write what you would tell both teams.

In [18]:
N_CLAIMS, FRAUD_RATE = 20_000, 0.02
# your investigation here

---
## Before you hand it in

- Did you print the **baseline** next to every accuracy?
- Did you quote precision **and** recall, and say which cut-off they came from?
- Did you pass `.predict_proba()[:, 1]` to every curve, not `.predict()`?
- Did you check the **support** before trusting a per-class rate?
- Did you say who chose the cut-off, and what each mistake costs?
- For any claim about subgroups, did you actually compute it **per subgroup**?

Five numbers worth carrying away:

- A readmission model at **89.3% accuracy** beat its do-nothing baseline by **1.3 points**
  and found **70 of 362** readmissions.
- Its precision of **0.7000** came from flagging 100 patients out of 3,000 — and rose to
  **0.7500** by flagging only 24.
- Its recall was **0.3363** for patients discharged to a nursing facility and **0.1129** for
  patients discharged home with care. One "weighted F1 of 0.86" reported that as consistent.
- The cost-optimal cut-off was **0.07**, not the 0.50 the model ships with, and worth
  about **₹768,000**.
- AUC computed by counting all **954,956** positive/negative pairs matched
  `roc_auc_score` to six decimal places — as long as ties counted as half a win.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*